In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./Data/RetailCleaned.csv')

In [3]:
# change datatype of 'InvoiceDate'
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [4]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2009-2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2009-2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010


In [5]:
df.shape

(1055238, 9)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055238 entries, 0 to 1055237
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1055238 non-null  object        
 1   StockCode    1055238 non-null  object        
 2   Description  1050856 non-null  object        
 3   Quantity     1055238 non-null  int64         
 4   InvoiceDate  1055238 non-null  datetime64[ns]
 5   Price        1055238 non-null  float64       
 6   Customer ID  812368 non-null   float64       
 7   Country      1055238 non-null  object        
 8   SourceSheet  1055238 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 72.5+ MB


In [7]:
# Filter and Calculate Recency, Frequency, Monetry (RFM)

# Filter valid customers and positive quantities
rfm_data = df[
    (df['Customer ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

# Reference date for Recency calculation (generally, today())
snapshot_date = pd.to_datetime('2011-12-10')

# Calculate total transaction amount
rfm_data['TotalAmount'] = rfm_data['Quantity'] * rfm_data['Price']

# Calculate RFM metrics for each customer
rfm = rfm_data.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),  # Days since last purchase
    Frequency=('Invoice', 'nunique'),                                  # Unique orders
    Monetary=('TotalAmount', 'sum')                                    # Total spending
).reset_index()


In [11]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [13]:
# Score customers and assign business segments

# Recency score   : recent purchases get higher scores
# Frequency score : more orders get higher scores
# Monetary score  : higher spending gets higher scores

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)


# Assign business segments based on RFM scores
def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)


# Flag customers inactive for more than 90 days
rfm['IsChurned'] = rfm['Recency'] > 90


In [14]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346.0,325,12,77556.46,2,5,5,At Risk,True
1,12347.0,2,8,5633.32,5,4,5,Champions,False
2,12348.0,75,5,2019.40,3,4,4,Loyal,False
3,12349.0,18,4,4428.69,5,3,5,Loyal,False
4,12350.0,310,1,334.40,2,1,2,Lost,True


In [18]:
# Merge RFM results back onto transcations

# Filter valid customers and positive quantities
df = df[(df['Customer ID'].notna()) & (df['Quantity'] > 0)].copy()

# Extract invoice month
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

# Calculate revenue
df['Revenue'] = df['Quantity'] * df['Price']


# Merge RFM metrics with transaction data
df_final = df.merge(rfm, on='Customer ID', how='left')


In [20]:
df_final.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2009-2010,2010-12-01,15.30,372,155,56600.08,2,5,5,At Risk,True
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010,2010-12-01,20.34,372,155,56600.08,2,5,5,At Risk,True
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2009-2010,2010-12-01,22.00,372,155,56600.08,2,5,5,At Risk,True
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010,2010-12-01,20.34,372,155,56600.08,2,5,5,At Risk,True
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2009-2010,2010-12-01,20.34,372,155,56600.08,2,5,5,At Risk,True


In [21]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 793680 entries, 0 to 793679
Data columns (total 19 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice       793680 non-null  object        
 1   StockCode     793680 non-null  object        
 2   Description   793680 non-null  object        
 3   Quantity      793680 non-null  int64         
 4   InvoiceDate   793680 non-null  datetime64[ns]
 5   Price         793680 non-null  float64       
 6   Customer ID   793680 non-null  float64       
 7   Country       793680 non-null  object        
 8   SourceSheet   793680 non-null  object        
 9   InvoiceMonth  793680 non-null  datetime64[ns]
 10  Revenue       793680 non-null  float64       
 11  Recency       793680 non-null  int64         
 12  Frequency     793680 non-null  int64         
 13  Monetary      793680 non-null  float64       
 14  R_Score       793680 non-null  int64         
 15  F_Score       793

In [19]:
# Export
df_final.to_csv('./Data/EcommerceDashboardData.csv', index=False)